In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import re, json, glob, math, random, pickle, hashlib, itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             f1_score, confusion_matrix, roc_curve, precision_recall_curve)
from scipy import stats

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

SEEDS = [42, 1, 7]
GLOBAL_SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 25
PATIENCE = 5

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
_MEAN_D = IMAGENET_MEAN.to(DEVICE)
_STD_D  = IMAGENET_STD.to(DEVICE)

def normalize_batch(x):
    return (x - _MEAN_D) / _STD_D

def set_determinism(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_worker(worker_id):
    ws = torch.initial_seed() % 2**32
    np.random.seed(ws); random.seed(ws)

set_determinism(GLOBAL_SEED)
print("Device:", DEVICE, "| torch", torch.__version__)
print("EPOCHS:", EPOCHS, "| PATIENCE:", PATIENCE, "| SEEDS:", SEEDS)

Device: cuda | torch 2.10.0+cu128
EPOCHS: 25 | PATIENCE: 5 | SEEDS: [42, 1, 7]


In [2]:
import torch.torch_version
torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])
print("safe globals registered")

safe globals registered


In [3]:
import os, shutil, glob
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
src = sorted(glob.glob("/kaggle/input/**/resnet18__*.pt", recursive=True))
print("found in inputs:", len(src))
for p in src:
    dst = os.path.join(WORK, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy2(p, dst)
print("in WORK:", len([f for f in os.listdir(WORK) if f.endswith(".pt")]), "checkpoints")

found in inputs: 15
in WORK: 15 checkpoints


In [4]:
PREFERRED = "/kaggle/input/datasets/ismailupal/cell-images-for-detecting-malaria/1.cell-images-for-detecting-malaria/cell_images"

def _valid(p):
    return (os.path.isdir(os.path.join(p, "Parasitized"))
            and os.path.isdir(os.path.join(p, "Uninfected")))

if _valid(PREFERRED):
    DATA_ROOT = PREFERRED
    print("Using preferred DATA_ROOT.")
else:
    cands = [c for c in sorted(glob.glob("/kaggle/input/**/cell_images", recursive=True)) if _valid(c)]
    assert cands, "cell_images/{Parasitized,Uninfected} not found under /kaggle/input"
    if len(cands) > 1:
        print("WARNING: multiple candidates, using the first:")
        for c in cands: print("   ", c)
    DATA_ROOT = cands[0]

print("DATA_ROOT:", DATA_ROOT)
n_para = len([f for f in os.listdir(os.path.join(DATA_ROOT, "Parasitized")) if f.lower().endswith(".png")])
n_unin = len([f for f in os.listdir(os.path.join(DATA_ROOT, "Uninfected"))  if f.lower().endswith(".png")])
print(f"Parasitized: {n_para} | Uninfected: {n_unin}   (expect 13779 / 13779)")
assert n_para == 13779 and n_unin == 13779, "Unexpected PNG counts"
assert n_para + n_unin == 27558

Using preferred DATA_ROOT.
DATA_ROOT: /kaggle/input/datasets/ismailupal/cell-images-for-detecting-malaria/1.cell-images-for-detecting-malaria/cell_images
Parasitized: 13779 | Uninfected: 13779   (expect 13779 / 13779)


In [5]:
def get_case_id(filename):
    m = re.search(r'^C(\d+)', filename)
    return int(m.group(1)) if m else None

def collect_samples(root):
    out = []
    for label, cls in enumerate(["Uninfected", "Parasitized"]):
        d = os.path.join(root, cls)
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith(".png"):
                out.append({"path": os.path.join(d, fn), "label": label,
                            "case": get_case_id(fn), "file": fn})
    return out

samples = collect_samples(DATA_ROOT)
assert sum(1 for s in samples if s["case"] is None) == 0, "unparseable case IDs"
cases = sorted({s["case"] for s in samples})
print(f"Total images: {len(samples)} | unique cases: {len(cases)}  (expect 27558 / 200)")

case_has_pos = {c: 0 for c in cases}
for s in samples:
    if s["label"] == 1:
        case_has_pos[s["case"]] = 1
strat = [case_has_pos[c] for c in cases]

train_c, temp_c = train_test_split(cases, test_size=0.30, random_state=GLOBAL_SEED, stratify=strat)
val_c, test_c = train_test_split(temp_c, test_size=0.50, random_state=GLOBAL_SEED,
                                 stratify=[case_has_pos[c] for c in temp_c])

def subset(cset):
    cset = set(cset)
    return [s for s in samples if s["case"] in cset]

train_samples, val_samples, test_samples = subset(train_c), subset(val_c), subset(test_c)

def split_hash(sset):
    h = hashlib.sha256()
    for f in sorted(s["file"] for s in sset):
        h.update(f.encode())
    return h.hexdigest()[:16]

SPLIT_HASHES = {n: split_hash(s) for n, s in
                [("train", train_samples), ("val", val_samples), ("test", test_samples)]}
for n, s in [("train", train_samples), ("val", val_samples), ("test", test_samples)]:
    pos = sum(x["label"] for x in s)
    print(f"{n}: {len(s)} imgs / {len(set(x['case'] for x in s))} cases | "
          f"{pos} parasitized / {len(s)-pos} uninfected | hash {SPLIT_HASHES[n]}")

EXPECTED_HASHES = {"train": "3ed101445b5fa25e", "val": "6c27f767fc6f2e59", "test": "45a6677551543a92"}
assert SPLIT_HASHES == EXPECTED_HASHES, f"SPLIT CHANGED: {SPLIT_HASHES} vs {EXPECTED_HASHES}"
print("Split matches the recorded hashes.")

pd.DataFrame([{"split": n, "file": s["file"], "label": s["label"], "case": s["case"]}
              for n, ss in [("train", train_samples), ("val", val_samples), ("test", test_samples)]
              for s in ss]).to_csv(os.path.join(WORK, "split_manifest.csv"), index=False)
json.dump(SPLIT_HASHES, open(os.path.join(WORK, "split_hashes.json"), "w"))
print("Saved manifest and hashes.")

Total images: 27558 | unique cases: 200  (expect 27558 / 200)
train: 18555 imgs / 140 cases | 8974 parasitized / 9581 uninfected | hash 3ed101445b5fa25e
val: 4878 imgs / 30 cases | 2756 parasitized / 2122 uninfected | hash 6c27f767fc6f2e59
test: 4125 imgs / 30 cases | 2049 parasitized / 2076 uninfected | hash 45a6677551543a92
Split matches the recorded hashes.
Saved manifest and hashes.


In [6]:
class MalariaDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples, self.transform = samples, transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["path"]).convert("RGB")
        return self.transform(img), s["label"]

class GreenOnly:
    def __call__(self, img):
        a = np.array(img)
        g = a[:, :, 1]
        return Image.fromarray(np.stack([g, g, g], axis=-1))

_BASE_GEOM = [T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(), T.RandomRotation(15)]

TRAIN_TF = {
    "baseline": T.Compose(_BASE_GEOM + [T.ToTensor()]),
    "colour_only": T.Compose(_BASE_GEOM + [
        T.RandomApply([T.Grayscale(num_output_channels=3)], p=0.25),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.4, hue=0.5),
        T.ToTensor()]),
    "colour_plus_blur": T.Compose(_BASE_GEOM + [
        T.RandomApply([T.Grayscale(num_output_channels=3)], p=0.25),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.4, hue=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.3),
        T.ToTensor()]),
    "gray_trained": T.Compose(_BASE_GEOM + [
        T.Grayscale(num_output_channels=3), T.ToTensor()]),
    "green_trained": T.Compose(_BASE_GEOM + [
        T.Lambda(lambda im: GreenOnly()(im)), T.ToTensor()]),
}

EVAL_TF_PLAIN = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])
EVAL_TF_GRAY  = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)),
                           T.Grayscale(num_output_channels=3), T.ToTensor()])
EVAL_TF_GREEN = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)),
                           T.Lambda(lambda im: GreenOnly()(im)), T.ToTensor()])

VARIANT_EVAL_TF = {
    "baseline": EVAL_TF_PLAIN, "colour_only": EVAL_TF_PLAIN, "colour_plus_blur": EVAL_TF_PLAIN,
    "gray_trained": EVAL_TF_GRAY, "green_trained": EVAL_TF_GREEN,
}
PROBE_VARIANTS = ["baseline", "colour_only", "colour_plus_blur"]

def make_loader(samples, transform, shuffle=False, bs=BATCH_SIZE, seed=GLOBAL_SEED):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(MalariaDataset(samples, transform), batch_size=bs, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
                      worker_init_fn=seed_worker, generator=g,
                      persistent_workers=(NUM_WORKERS > 0))

val_loader_plain = make_loader(val_samples, EVAL_TF_PLAIN)
val_loader_gray  = make_loader(val_samples, EVAL_TF_GRAY)
val_loader_green = make_loader(val_samples, EVAL_TF_GREEN)
VARIANT_VAL_LOADER = {"baseline": val_loader_plain, "colour_only": val_loader_plain,
                      "colour_plus_blur": val_loader_plain,
                      "gray_trained": val_loader_gray, "green_trained": val_loader_green}
print("Loaders ready.")

Loaders ready.


In [7]:
class Grayscale3Ch:
    def __call__(self, img): return TF.to_grayscale(img, num_output_channels=3)

class HueRotate:
    def __init__(self, degrees): self.factor = degrees / 360.0
    def __call__(self, img): return TF.adjust_hue(img, self.factor)

class ChannelPermute:
    def __init__(self, order): self.order = order
    def __call__(self, img): return Image.fromarray(np.array(img)[:, :, self.order])

class StrongColorJitter:
    def __init__(self):
        self.j = T.ColorJitter(brightness=0.4, contrast=0.3, saturation=0.5, hue=0.15)
    def __call__(self, img): return self.j(img)

class GaussianBlurControl:
    def __init__(self, kernel_size=9, sigma=2.5):
        self.b = T.GaussianBlur(kernel_size=kernel_size, sigma=sigma)
    def __call__(self, img): return self.b(img)

class GaussianNoiseControl:
    def __init__(self, std=0.08, seed=GLOBAL_SEED): self.std, self.seed = std, seed
    def __call__(self, t):
        g = torch.Generator(device=t.device); g.manual_seed(self.seed)
        n = torch.randn(t.shape, generator=g, device=t.device, dtype=t.dtype) * self.std
        return torch.clamp(t + n, 0.0, 1.0)

class ZeroChannel:
    def __init__(self, idx): self.idx = idx
    def __call__(self, img):
        a = np.array(img).copy(); a[:, :, self.idx] = 0
        return Image.fromarray(a)

class KeepOnlyChannel:
    def __init__(self, idx): self.idx = idx
    def __call__(self, img):
        a = np.array(img).copy()
        for i in range(3):
            if i != self.idx: a[:, :, i] = 0
        return Image.fromarray(a)

PIL_INTERVENTIONS = {
    "grayscale":           Grayscale3Ch(),
    "hue_30":              HueRotate(30),
    "hue_60":              HueRotate(60),
    "hue_90":              HueRotate(90),
    "channel_BGR":         ChannelPermute([2, 1, 0]),
    "channel_GBR":         ChannelPermute([1, 2, 0]),
    "channel_GRB":         ChannelPermute([1, 0, 2]),
    "color_jitter_strong": StrongColorJitter(),
    "blur_control":        GaussianBlurControl(),
}
TENSOR_INTERVENTIONS = {"noise_control": GaussianNoiseControl(std=0.08)}

ABLATIONS = {f"zero_{c}": ZeroChannel(i) for i, c in enumerate("RGB")}
ABLATIONS.update({f"only_{c}": KeepOnlyChannel(i) for i, c in enumerate("RGB")})

HEADLINE = ["clean", "grayscale", "hue_90", "channel_GRB", "channel_GBR",
            "channel_BGR", "blur_control", "noise_control"]
print("Interventions:", list(PIL_INTERVENTIONS) + list(TENSOR_INTERVENTIONS))
print("Ablations:", list(ABLATIONS))

Interventions: ['grayscale', 'hue_30', 'hue_60', 'hue_90', 'channel_BGR', 'channel_GBR', 'channel_GRB', 'color_jitter_strong', 'blur_control', 'noise_control']
Ablations: ['zero_R', 'zero_G', 'zero_B', 'only_R', 'only_G', 'only_B']


In [8]:
def ece_equal_width(labels, probs, preds, n_bins=15):
    conf = np.maximum(probs, 1 - probs)
    correct = (preds == labels).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece, stats_out = 0.0, []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0:
            stats_out.append((lo, hi, np.nan, np.nan, 0)); continue
        a, c = correct[m].mean(), conf[m].mean()
        ece += (m.sum() / len(labels)) * abs(a - c)
        stats_out.append((lo, hi, a, c, int(m.sum())))
    return ece, stats_out

def ece_equal_mass(labels, probs, preds, n_bins=15):
    conf = np.maximum(probs, 1 - probs)
    correct = (preds == labels).astype(float)
    order = np.argsort(conf)
    ece = 0.0
    for chunk in np.array_split(order, n_bins):
        if len(chunk) == 0: continue
        ece += (len(chunk) / len(labels)) * abs(correct[chunk].mean() - conf[chunk].mean())
    return ece

def brier(labels, probs):
    return float(np.mean((probs - labels) ** 2))

def precision_at_sensitivity(labels, probs, target=0.90):
    prec, rec, _ = precision_recall_curve(labels, probs)
    ok = rec >= target
    return float(prec[ok].max()) if ok.any() else float("nan")

def metric_pack(labels, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    out = {
        "threshold": threshold,
        "accuracy": float((preds == labels).mean()),
        "sensitivity": float(tp / (tp + fn)) if (tp + fn) else np.nan,
        "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
        "precision": float(precision_score(labels, preds, zero_division=0)),
        "f1": float(f1_score(labels, preds, zero_division=0)),
        "mean_confidence": float(np.mean(np.maximum(probs, 1 - probs))),
        "brier": brier(labels, probs),
    }
    out["ece"], _ = ece_equal_width(labels, probs, preds)
    out["ece_adaptive"] = ece_equal_mass(labels, probs, preds)
    try:
        out["auroc"] = float(roc_auc_score(labels, probs))
        out["auprc"] = float(average_precision_score(labels, probs))
        out["prec_at_sens90"] = precision_at_sensitivity(labels, probs, 0.90)
    except ValueError:
        out["auroc"] = out["auprc"] = out["prec_at_sens90"] = np.nan
    return out

def bootstrap_ci(labels, probs, fn, n_boot=1000, seed=GLOBAL_SEED, alpha=0.05):
    rng = np.random.default_rng(seed)
    n = len(labels); vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        try: vals.append(fn(labels[idx], probs[idx]))
        except Exception: pass
    if not vals: return (np.nan, np.nan)
    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)
print("Metrics ready.")

Metrics ready.


In [9]:
def build_resnet18():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

def build_mobilenetv3():
    m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 2)
    return m.to(DEVICE)

BUILDERS = {"resnet18": build_resnet18, "mobilenetv3": build_mobilenetv3}

def ckpt_path(arch, variant, seed):
    return os.path.join(WORK, f"{arch}__{variant}__seed{seed}.pt")

def build_meta(arch, variant, seed, best_val_loss, epochs_run):
    return {"arch": arch, "variant": variant, "seed": seed,
            "split_hash_train": SPLIT_HASHES["train"],
            "split_hash_val": SPLIT_HASHES["val"],
            "split_hash_test": SPLIT_HASHES["test"],
            "normalization": "imagenet_mean_std",
            "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
            "epochs_max": EPOCHS, "epochs_run": epochs_run, "patience": PATIENCE,
            "best_val_loss": float(best_val_loss),
            "amp": True, "cudnn_deterministic": True,
            "torch": str(torch.__version__), "code_version": "v3"}

def check_meta(meta, arch, variant, seed):
    if meta is None:
        return ["no metadata (pre-v3 checkpoint)"]
    bad = []
    for k, want in [("arch", arch), ("variant", variant), ("seed", seed),
                    ("split_hash_train", SPLIT_HASHES["train"]),
                    ("split_hash_test", SPLIT_HASHES["test"]),
                    ("normalization", "imagenet_mean_std"),
                    ("img_size", IMG_SIZE), ("epochs_max", EPOCHS),
                    ("code_version", "v3")]:
        if meta.get(k) != want:
            bad.append(f"{k}: file={meta.get(k)!r} expected={want!r}")
    return bad

def run_epoch(model, loader, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot_loss = correct = total = 0
    crit = nn.CrossEntropyLoss()
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs = normalize_batch(imgs.to(DEVICE, non_blocking=True))
            labels = labels.to(DEVICE, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                    out = model(imgs); loss = crit(out, labels)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                out = model(imgs); loss = crit(out, labels)
            tot_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return tot_loss / total, correct / total

def train_one(arch, variant, seed, epochs=None, patience=None, force=False):
    epochs = EPOCHS if epochs is None else epochs
    patience = PATIENCE if patience is None else patience
    path = ckpt_path(arch, variant, seed)
    if os.path.exists(path) and not force:
        try:
            blob = torch.load(path, map_location="cpu")
            bad = check_meta(blob.get("meta") if isinstance(blob, dict) else None,
                             arch, variant, seed)
        except Exception as e:
            bad = [f"unreadable: {e}"]
        if not bad:
            print(f"[skip] {os.path.basename(path)} verified on disk"); return path
        print(f"[RETRAIN] {os.path.basename(path)} failed provenance:")
        for b in bad: print("    ", b)

    set_determinism(seed)
    model = BUILDERS[arch]()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.1, patience=2)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    tr_loader = make_loader(train_samples, TRAIN_TF[variant], shuffle=True, seed=seed)
    va_loader = VARIANT_VAL_LOADER[variant]
    best, ctr, ep_done = float("inf"), 0, 0
    import time; t0 = time.time()
    for ep in range(epochs):
        trl, tra = run_epoch(model, tr_loader, opt, scaler)
        vll, vla = run_epoch(model, va_loader)
        sched.step(vll); ep_done = ep + 1
        print(f"  {arch}/{variant}/s{seed} ep{ep+1:02d}: train_acc={tra:.4f} "
              f"val_loss={vll:.4f} val_acc={vla:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if vll < best:
            best, ctr = vll, 0
            torch.save({"state_dict": model.state_dict(),
                        "meta": build_meta(arch, variant, seed, best, ep_done)}, path)
        else:
            ctr += 1
            if ctr >= patience:
                print("  early stop"); break
    print(f"[saved] {os.path.basename(path)} (best val_loss {best:.4f}, {time.time()-t0:.0f}s)", flush=True)
    del model; torch.cuda.empty_cache()
    return path

def discover_checkpoints(arch=None, verify=True):
    reg, rejected = {}, []
    for p in sorted(glob.glob(os.path.join(WORK, "*__*__seed*.pt"))):
        b = os.path.basename(p)[:-3]
        a, v, s = b.split("__"); s = int(s.replace("seed", ""))
        if arch and a != arch: continue
        if verify:
            try:
                blob = torch.load(p, map_location="cpu")
                bad = check_meta(blob.get("meta") if isinstance(blob, dict) else None, a, v, s)
            except Exception as e:
                bad = [f"unreadable: {e}"]
            if bad:
                rejected.append((os.path.basename(p), bad)); continue
        reg[(a, v, s)] = p
    if rejected:
        print("REJECTED checkpoints:")
        for n, bad in rejected: print(f"  {n}: {bad}")
    return reg
print("Training utilities ready.")

Training utilities ready.


In [10]:
def predict(model, samples, base_tf=EVAL_TF_PLAIN, pil_interv=None, tensor_interv=None,
            bs=BATCH_SIZE):
    tf_list = [T.Resize((IMG_SIZE, IMG_SIZE))]
    if base_tf is EVAL_TF_GRAY:  tf_list.append(T.Grayscale(num_output_channels=3))
    if base_tf is EVAL_TF_GREEN: tf_list.append(T.Lambda(lambda im: GreenOnly()(im)))
    if pil_interv is not None:   tf_list.append(T.Lambda(lambda im: pil_interv(im)))
    tf_list.append(T.ToTensor())
    loader = make_loader(samples, T.Compose(tf_list), shuffle=False, bs=bs)
    model.eval()
    L, P = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            if tensor_interv is not None: imgs = tensor_interv(imgs)
            probs = torch.softmax(model(normalize_batch(imgs)), dim=1)[:, 1]
            L.append(labels.numpy()); P.append(probs.float().cpu().numpy())
    return np.concatenate(L), np.concatenate(P)

def load_model(arch, path, strict_meta=True):
    blob = torch.load(path, map_location=DEVICE)
    if isinstance(blob, dict) and "state_dict" in blob:
        sd, meta = blob["state_dict"], blob.get("meta")
    else:
        sd, meta = blob, None
    b = os.path.basename(path)[:-3].split("__")
    bad = check_meta(meta, b[0], b[1], int(b[2].replace("seed", "")))
    if bad:
        msg = f"provenance mismatch for {os.path.basename(path)}: {bad}"
        if strict_meta: raise AssertionError(msg)
        print("WARNING:", msg)
    m = BUILDERS[arch]()
    m.load_state_dict(sd)
    m.eval()
    return m

def ckpt_meta(path):
    blob = torch.load(path, map_location="cpu")
    return blob.get("meta") if isinstance(blob, dict) else None
print("Evaluation core ready.")

Evaluation core ready.


In [11]:
_required = ["DATA_ROOT","train_samples","val_samples","test_samples","SPLIT_HASHES",
             "TRAIN_TF","VARIANT_EVAL_TF","VARIANT_VAL_LOADER","make_loader",
             "PIL_INTERVENTIONS","TENSOR_INTERVENTIONS","ABLATIONS",
             "metric_pack","bootstrap_ci","brier","ece_equal_width","ece_equal_mass",
             "BUILDERS","train_one","discover_checkpoints","ckpt_path","build_meta",
             "check_meta","run_epoch","predict","load_model","ckpt_meta",
             "normalize_batch","set_determinism","seed_worker"]
_missing = [n for n in _required if n not in globals()]
assert not _missing, f"Stage A incomplete. Missing: {_missing}"

print("Stage A complete.")
print("split hashes:", SPLIT_HASHES)
print(f"train {len(train_samples)} | val {len(val_samples)} | test {len(test_samples)}")

_m = BUILDERS["resnet18"]()
_y, _p = predict(_m, test_samples[:128])
print(f"smoke test OK — {len(_p)} predictions, prob range [{_p.min():.3f}, {_p.max():.3f}]")
del _m; torch.cuda.empty_cache()

_m = BUILDERS["resnet18"]()
_y, _p = predict(_m, test_samples[:128], base_tf=EVAL_TF_GREEN)
print(f"green-path smoke test OK — prob range [{_p.min():.3f}, {_p.max():.3f}]")
del _m; torch.cuda.empty_cache()

Stage A complete.
split hashes: {'train': '3ed101445b5fa25e', 'val': '6c27f767fc6f2e59', 'test': '45a6677551543a92'}
train 18555 | val 4878 | test 4125
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 209MB/s]


smoke test OK — 128 predictions, prob range [0.404, 0.757]
green-path smoke test OK — prob range [0.219, 0.572]


In [12]:
import time; t0 = time.time()
for variant in ["baseline", "colour_only", "colour_plus_blur"]:
    for seed in SEEDS:
        try:
            train_one("resnet18", variant, seed)
        except Exception as e:
            print(f"!! FAILED {variant}/seed{seed}: {type(e).__name__}: {e}", flush=True)
        print(f"--- total elapsed {(time.time()-t0)/60:.1f} min ---", flush=True)
print("\nOn disk:", sorted(os.path.basename(p) for p in discover_checkpoints("resnet18").values()))

[skip] resnet18__baseline__seed42.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__baseline__seed1.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__baseline__seed7.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__colour_only__seed42.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__colour_only__seed1.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__colour_only__seed7.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__colour_plus_blur__seed42.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__colour_plus_blur__seed1.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__colour_plus_blur__seed7.pt verified on disk
--- total elapsed 0.0 min ---

On disk: ['resnet18__baseline__seed1.pt', 'resnet18__baseline__seed42.pt', 'resnet18__baseline__seed7.pt', 'resnet18__colour_only__seed1.pt', 'resnet18__colour_only__seed42.pt', 'resnet18__colour_only__seed7.pt', 're

In [13]:
import time; t0 = time.time()
for variant in ["gray_trained", "green_trained"]:
    for seed in SEEDS:
        try:
            train_one("resnet18", variant, seed)
        except Exception as e:
            print(f"!! FAILED {variant}/seed{seed}: {type(e).__name__}: {e}", flush=True)
        print(f"--- total elapsed {(time.time()-t0)/60:.1f} min ---", flush=True)

REG = discover_checkpoints("resnet18")
print(f"\nverified checkpoints: {len(REG)} (expect 15)")
for k, p in sorted(REG.items()):
    m = ckpt_meta(p)
    print(f"  {k}: epochs_run={m['epochs_run']} best_val_loss={m['best_val_loss']:.4f}")
print("\nfiles in WORK:", sorted(f for f in os.listdir(WORK) if f.endswith((".pt", ".csv", ".json"))))

[skip] resnet18__gray_trained__seed42.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__gray_trained__seed1.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__gray_trained__seed7.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__green_trained__seed42.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__green_trained__seed1.pt verified on disk
--- total elapsed 0.0 min ---
[skip] resnet18__green_trained__seed7.pt verified on disk
--- total elapsed 0.0 min ---

verified checkpoints: 15 (expect 15)
  ('resnet18', 'baseline', 1): epochs_run=14 best_val_loss=0.0728
  ('resnet18', 'baseline', 7): epochs_run=18 best_val_loss=0.0695
  ('resnet18', 'baseline', 42): epochs_run=18 best_val_loss=0.0726
  ('resnet18', 'colour_only', 1): epochs_run=17 best_val_loss=0.0807
  ('resnet18', 'colour_only', 7): epochs_run=18 best_val_loss=0.0761
  ('resnet18', 'colour_only', 42): epochs_run=16 best_val_loss=0.0820
  ('resnet18', 'colour_plus_

In [14]:
# ============ CELL C1 — in-domain probe ============
REG = discover_checkpoints()
print(f"Found {len(REG)} checkpoints.")

CONDITIONS = [("clean", None, None)] + \
             [(k, v, None) for k, v in PIL_INTERVENTIONS.items()] + \
             [(k, None, v) for k, v in TENSOR_INTERVENTIONS.items()]

master_rows, raw_store = [], {}
for (arch, variant, seed), path in sorted(REG.items()):
    m = load_model(arch, path)
    base_tf = VARIANT_EVAL_TF[variant]
    conds = CONDITIONS if variant in PROBE_VARIANTS else [("clean", None, None)]
    for name, pil_i, ten_i in conds:
        y, p = predict(m, test_samples, base_tf=base_tf, pil_interv=pil_i, tensor_interv=ten_i)
        row = {"architecture": arch, "variant": variant, "seed": seed, "condition": name}
        row.update(metric_pack(y, p))
        master_rows.append(row)
        raw_store[(arch, variant, seed, name)] = (y, p)
    print(f"done {arch}/{variant}/s{seed}", flush=True)
    del m; torch.cuda.empty_cache()

master_df = pd.DataFrame(master_rows)
master_df.to_csv(os.path.join(WORK, "MASTER_indomain.csv"), index=False)
with open(os.path.join(WORK, "MASTER_indomain_raw.pkl"), "wb") as f:
    pickle.dump({f"{a}|{v}|{s}|{c}": (y, p) for (a, v, s, c), (y, p) in raw_store.items()}, f)

summ = (master_df.groupby(["architecture", "variant", "condition"])
        [["accuracy", "sensitivity", "specificity", "auroc", "ece", "ece_adaptive", "brier"]]
        .agg(["mean", "std"]).round(4))
summ.to_csv(os.path.join(WORK, "MASTER_indomain_summary.csv"))
for arch in master_df["architecture"].unique():
    for variant in PROBE_VARIANTS:
        idx = summ.index.droplevel(2).unique()
        if (arch, variant) not in idx: continue
        sub = summ.loc[(arch, variant)]
        print(f"\n=== {arch} / {variant} ===")
        print(sub.loc[[c for c in HEADLINE if c in sub.index]].to_string())

Found 15 checkpoints.
done resnet18/baseline/s1
done resnet18/baseline/s7
done resnet18/baseline/s42
done resnet18/colour_only/s1
done resnet18/colour_only/s7
done resnet18/colour_only/s42
done resnet18/colour_plus_blur/s1
done resnet18/colour_plus_blur/s7
done resnet18/colour_plus_blur/s42
done resnet18/gray_trained/s1
done resnet18/gray_trained/s7
done resnet18/gray_trained/s42
done resnet18/green_trained/s1
done resnet18/green_trained/s7
done resnet18/green_trained/s42

=== resnet18 / baseline ===
              accuracy         sensitivity         specificity           auroc             ece         ece_adaptive           brier        
                  mean     std        mean     std        mean     std    mean     std    mean     std         mean     std    mean     std
condition                                                                                                                                  
clean           0.9699  0.0020      0.9665  0.0044      0.9732  0.0010  0.

In [15]:
# ============ CELL C2 — the decisive control table ============
ctrl = master_df[(master_df["condition"] == "clean") & (master_df["architecture"] == "resnet18")]
ctrl_summary = (ctrl.groupby("variant")[["accuracy", "sensitivity", "specificity", "auroc", "ece"]]
                .agg(["mean", "std"]).round(4))
print("=== CLEAN in-domain performance by training variant (ResNet-18) ===")
print(ctrl_summary.to_string())
ctrl_summary.to_csv(os.path.join(WORK, "C2_control_models_clean.csv"))

gray_acc = ctrl[ctrl["variant"] == "gray_trained"]["accuracy"]
base_acc = ctrl[ctrl["variant"] == "baseline"]["accuracy"]
if len(gray_acc) and len(base_acc):
    gap = base_acc.mean() - gray_acc.mean()
    print(f"\ngray_trained mean acc = {gray_acc.mean():.4f} | "
          f"baseline = {base_acc.mean():.4f} | gap = {gap:+.4f}")
    print("VERDICT:", "colour is NOT necessary -> over-reliance framing is earned (gap small)"
          if gap < 0.03 else
          "colour carries real signal -> reframe (gap large)")

print("\n95% bootstrap CIs on clean accuracy (seed 42):")
for variant in TRAIN_TF:
    key = ("resnet18", variant, 42, "clean")
    if key in raw_store:
        y, p = raw_store[key]
        acc = ((p >= 0.5).astype(int) == y).mean()
        lo, hi = bootstrap_ci(y, p, lambda l, q: float(((q >= 0.5).astype(int) == l).mean()))
        print(f"  {variant:18s} {acc:.4f}  [{lo:.4f}, {hi:.4f}]")

=== CLEAN in-domain performance by training variant (ResNet-18) ===
                 accuracy         sensitivity         specificity           auroc             ece        
                     mean     std        mean     std        mean     std    mean     std    mean     std
variant                                                                                                  
baseline           0.9699  0.0020      0.9665  0.0044      0.9732  0.0010  0.9949  0.0002  0.0088  0.0026
colour_only        0.9653  0.0010      0.9623  0.0020      0.9682  0.0005  0.9941  0.0005  0.0072  0.0027
colour_plus_blur   0.9668  0.0017      0.9655  0.0032      0.9680  0.0064  0.9942  0.0001  0.0095  0.0020
gray_trained       0.9664  0.0015      0.9694  0.0016      0.9634  0.0017  0.9943  0.0002  0.0087  0.0016
green_trained      0.9674  0.0009      0.9694  0.0054      0.9653  0.0038  0.9940  0.0003  0.0083  0.0002

gray_trained mean acc = 0.9664 | baseline = 0.9699 | gap = +0.0035
VERDICT: colour 